### VPC Flow Log Hunting: part of the DUNE project
Ingests VPC flow logs, labels known cloud provider IPs, tor IPs, and anything known you want to define in known_ips.txt. Flows are then aggregated into records with summed time spans, byte and packet counts. Flows are typed as north-south and east-west according to direction. At the end of the notebook below, three interfaces exist for hunting:
1. Flows counted by total bytes or time duration
2. Flows counted by time duration (for comparison)
3. Beaconing candidates - flows with regular frequency and byte counts
   
Field definitions:

type: north-south (involving external networks) or east-west (staying within private networks)

direction: ingress or egress if populated; derived for older flow records.

action: accept or reject, according to the flow records

label: the name of the cloud provider, if any, the flow associates with


In [ ]:
# --- cell 00 ---
import pandas as pd
import ipaddress
from ipaddress import ip_address
import numpy as np
import os
import json
from functools import reduce
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# plotly 6 FigureWidget: after a redraw the front end can report a trace delta with
# no uid, and plotly's handler dies with KeyError: 'uid'. Fill the uid in by trace
# position before plotly reads it. Guarded so re-running this cell doesn't double-wrap.

from plotly.basewidget import BaseFigureWidget
if not getattr(BaseFigureWidget, "_uid_patch", False):
    _orig_trace_deltas = BaseFigureWidget._handler_js2py_traceDeltas.func
    def _trace_deltas_with_uid(self, change):
        for i, d in enumerate((change["new"] or {}).get("trace_deltas") or []):
            if "uid" not in d and i < len(self.data):
                d["uid"] = self.data[i].uid
        return _orig_trace_deltas(self, change)
    BaseFigureWidget._handler_js2py_traceDeltas = _trace_deltas_with_uid
    BaseFigureWidget._uid_patch = True

In [ ]:
# --- cell 01 ---
# All reference CIDR data lives here (local snapshots, no network needed)

CIDR_DIR   = "cidrs"
_REF_COLS  = ["ip_prefix", "region", "service", "network_border_group"]

# Load AWS IP ranges from the local snapshot in cidrs/.
# Missing / unreadable file -> warn and carry on with an empty frame.

try:
    with open(os.path.join(CIDR_DIR, "ip-ranges.json")) as _f:
        aws_data = json.load(_f)
    aws = pd.DataFrame(aws_data["prefixes"])[_REF_COLS]
except (OSError, ValueError, KeyError) as _e:
    print(f"!! AWS ip-ranges.json not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    aws = pd.DataFrame(columns=_REF_COLS)

aws

In [ ]:
# --- cell 02 ---
# Load the already-normalized Zscaler CIDR list from cidrs/zscaler.csv.
# Schema matches `aws`: ip_prefix, region, service, network_border_group, network.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
try:
    zscaler = pd.read_csv(os.path.join(CIDR_DIR, "zscaler.csv"))
    zscaler["network"] = zscaler["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
except (OSError, ValueError, KeyError) as _e:
    print(f"!! zscaler.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    zscaler = pd.DataFrame(columns=_SCHEMA)

print(f"{len(zscaler)} Zscaler CIDRs")
zscaler

In [ ]:
# --- cell 03 ---
# Load the already-normalized Tor exit list from cidrs/tor.csv.
# Schema matches `aws`; each entry is a /32 host. service = "TOR".
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
try:
    tor = pd.read_csv(os.path.join(CIDR_DIR, "tor.csv"))
    tor["network"] = tor["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
except (OSError, ValueError, KeyError) as _e:
    print(f"!! tor.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    tor = pd.DataFrame(columns=_SCHEMA)

print(f"{len(tor)} Tor exit IPs")
tor

In [ ]:
# --- cell 04 ---
# Azure Service Tags (cidrs/azure.json) -> same schema as `aws`.
# Structure differs from AWS: values[] = service tags, CIDRs nested in
# properties.addressPrefixes[] -> explode. service = systemService, falling back
# to the tag `name` when systemService is blank. IPv6 dropped. One row per CIDR:
# a specific tag beats the AzureCloud umbrella, and a tag with a region beats one without.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
try:
    with open(os.path.join(CIDR_DIR, "azure.json")) as _f:
        _az = json.load(_f)

    _rows = []
    for _v in _az["values"]:
        _p   = _v["properties"]
        _svc = _p.get("systemService") or _v["name"]
        _reg = _p.get("region") or ""
        for _pfx in _p.get("addressPrefixes", []):
            if ":" in _pfx:                       # IPv4 only
                continue
            _rows.append({"ip_prefix": _pfx, "region": _reg, "service": _svc,
                          "_umbrella": _v["name"].startswith("AzureCloud")})

    azure = (pd.DataFrame(_rows)
             .sort_values(["ip_prefix", "_umbrella", "region"],
                          ascending=[True, True, False], kind="stable")
             .drop_duplicates("ip_prefix", keep="first")
             .drop(columns="_umbrella")
             .reset_index(drop=True))
    azure["network_border_group"] = pd.NA
    azure["region"]  = azure["region"].replace("", pd.NA)
    azure["network"] = azure["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
    azure = azure[_SCHEMA]
    _chg = _az.get("changeNumber")
except (OSError, ValueError, KeyError) as _e:
    print(f"!! azure.json not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    azure, _chg = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(azure)} Azure CIDRs  ({azure['service'].nunique()} services, changeNumber {_chg})")
azure

In [ ]:
# --- cell 05 ---
# Google ranges: goog.json (all Google) + gcloud.json (GCP customer ranges, a
# subset of goog) -> same schema as `aws`. IPv4 only. service/region come from
# gcloud; the goog-only remainder is a generic "GOOGLE". A specific gcloud prefix
# beats a broad goog block via most-specific-match downstream; an exact-duplicate
# prefix keeps the gcloud row (it carries service + region).
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
try:
    with open(os.path.join(CIDR_DIR, "gcloud.json")) as _f:
        _gc = json.load(_f)
    with open(os.path.join(CIDR_DIR, "goog.json")) as _f:
        _gg = json.load(_f)

    _rows = [{"ip_prefix": _p["ipv4Prefix"], "region": _p.get("scope"),
              "service": _p.get("service") or "Google Cloud", "_src": "gcloud"}
             for _p in _gc["prefixes"] if "ipv4Prefix" in _p]
    _rows += [{"ip_prefix": _p["ipv4Prefix"], "region": pd.NA,
               "service": "GOOGLE", "_src": "goog"}
              for _p in _gg["prefixes"] if "ipv4Prefix" in _p]

    google = (pd.DataFrame(_rows)
              .sort_values("_src", kind="stable")          # gcloud rows before goog
              .drop_duplicates("ip_prefix", keep="first")
              .drop(columns="_src")
              .reset_index(drop=True))
    google["network_border_group"] = pd.NA
    google["region"]  = google["region"].replace("", pd.NA)
    google["network"] = google["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
    google = google[_SCHEMA]
    _sync = _gc.get("syncToken")
except (OSError, ValueError, KeyError) as _e:
    print(f"!! Google ranges not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    google, _sync = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(google)} Google CIDRs  "
      f"({int((google['service'] == 'Google Cloud').sum())} GCP, "
      f"{int((google['service'] == 'GOOGLE').sum())} other Google, syncToken {_sync})")
google

In [ ]:
# --- cell 06 ---
# load flow data

parquet_file = "vflows.parquet"
vflows= pd.read_parquet(parquet_file)
vflows.shape

In [ ]:
# --- cell 07 ---
# Duplicate rows in vflows, matched on DUP_COLS only. Comment a column out to
# ignore it, e.g. drop interface-id to catch the same flow logged by two ENIs.

DUP_COLS = [
    "version",
    "account-id",
    #"interface-id",
    "srcaddr",
    "dstaddr",
    "srcport",
    "dstport",
    "protocol",
    #"packets",
    #"bytes",
    #"start",
    #"end",
    # "start_time",    # same instant as start
    # "end_time",      # same instant as end
    "action",
    "log-status",
]

_missing = [c for c in DUP_COLS if c not in vflows.columns]
if _missing:
    raise KeyError(f"not in vflows: {_missing}")

_dup   = vflows.duplicated(subset=DUP_COLS, keep=False)
_extra = int(vflows.duplicated(subset=DUP_COLS, keep="first").sum())

print(f"matching on {len(DUP_COLS)} columns: {', '.join(DUP_COLS)}")
print(f"{len(vflows):,} rows")
print(f"{_extra:,} extra copies (rows you'd drop to dedupe)  {_extra / len(vflows):.2%}")
print(f"{int(_dup.sum()):,} rows involved in duplication, "
      f"{int(_dup.sum()) - _extra:,} distinct duplicated records")

vflowsDups = (vflows[_dup].groupby(DUP_COLS, dropna=False).size()
               .rename("copies").sort_values(ascending=False).reset_index())
vflowsDups.head(20)

In [ ]:
# --- cell 08 ---
# Aggregate repeated flow records into sessions -> vflowsAgg (vflows untouched).
# Key = DUP_COLS from the cell above, minus the per-record counters and times.
# Records with the same key merge while each starts within MERGE_GAP seconds of
# the latest end so far; a longer gap starts a new session, so repeat connections
# (scans, beacons, ICMP, reused source ports) keep their timing.

MERGE_GAP = 600   # seconds -- AWS's longest flow-log aggregation window

_per_rec = ["packets", "bytes", "start", "end", "start_time", "end_time"]
_key     = [c for c in DUP_COLS if c not in _per_rec]

_v   = vflows[[c for c in vflows.columns if c not in ("direction", "type")]]
_gid = _v.groupby(_key, dropna=False, sort=False).ngroup()
_v   = _v.assign(_gid=_gid).sort_values(["_gid", "start"], kind="stable")

_prev_end = _v.groupby("_gid")["end"].cummax().groupby(_v["_gid"]).shift()
_new      = _v["_gid"].ne(_v["_gid"].shift()) | (_v["start"] - _prev_end).gt(MERGE_GAP)
_v["_sid"] = _new.cumsum()

_aggs = {c: (c, "first") for c in _v.columns if c not in _per_rec + ["_gid", "_sid"]}
_aggs.update(start=("start", "min"), end=("end", "max"),
             start_time=("start_time", "min"), end_time=("end_time", "max"),
             records=("start", "size"))

_g   = _v.groupby("_sid", sort=False)
_out = _g.agg(**_aggs)
_out[["packets", "bytes"]] = _g[["packets", "bytes"]].sum(min_count=1)   # NODATA stays <NA>, not 0

vflowsAgg = (_out[[c for c in vflows.columns if c in _out.columns] + ["records"]]
              .sort_values("start", kind="stable")
              .reset_index(drop=True))

_merged = int((vflowsAgg["records"] > 1).sum())
print(f"key ({len(_key)}): {', '.join(_key)}   gap <= {MERGE_GAP}s")
print(f"{len(vflows):,} records -> {len(vflowsAgg):,} sessions  "
      f"({len(vflows) - len(vflowsAgg):,} folded in, {_merged:,} sessions built from >1 record)")
print(f"bytes   {vflows['bytes'].sum():,} -> {vflowsAgg['bytes'].sum():,}")
print(f"packets {vflows['packets'].sum():,} -> {vflowsAgg['packets'].sum():,}")
vflowsAgg.sort_values("records", ascending=False).head(10)

In [ ]:
# --- cell 09 ---
# unique public IPs across ALL flows (either end). Vectorised end to end:
# parse every distinct address once, classify by integer range (no per-IP Python
# / ipaddress calls), and keep the int for the CIDR match downstream.

def ipv4_to_int(s):
    o = pd.Series(s, dtype="string").str.split(".", expand=True).astype("uint64")
    return (o[0] << np.uint64(24) | o[1] << np.uint64(16)
            | o[2] << np.uint64(8) | o[3]).to_numpy("uint64")

# IPv4 blocks ipaddress marks non-global (RFC1918, CGNAT, loopback, link-local,
# doc/benchmark ranges, multicast + reserved + broadcast)

_NON_GLOBAL = [
    ("0.0.0.0", "0.255.255.255"),       ("10.0.0.0", "10.255.255.255"),
    ("100.64.0.0", "100.127.255.255"),  ("127.0.0.0", "127.255.255.255"),
    ("169.254.0.0", "169.254.255.255"), ("172.16.0.0", "172.31.255.255"),
    ("192.0.0.0", "192.0.0.255"),       ("192.0.2.0", "192.0.2.255"),
    ("192.88.99.0", "192.88.99.255"),   ("192.168.0.0", "192.168.255.255"),
    ("198.18.0.0", "198.19.255.255"),   ("198.51.100.0", "198.51.100.255"),
    ("203.0.113.0", "203.0.113.255"),   ("224.0.0.0", "255.255.255.255"),
]

_ips  = pd.unique(np.concatenate([vflowsAgg["srcaddr"].unique(), vflowsAgg["dstaddr"].unique()]))
_ips  = _ips[pd.Series(_ips).str.count(r"\.").eq(3).to_numpy()]     # drop None / IPv6
_ints = ipv4_to_int(_ips)

_pub = np.ones(len(_ips), dtype=bool)
for _lo, _hi in _NON_GLOBAL:
    _pub &= ~((_ints >= ipv4_to_int([_lo])[0]) & (_ints <= ipv4_to_int([_hi])[0]))

_ord   = np.argsort(_ints[_pub], kind="stable")
public = pd.DataFrame({"ip": _ips[_pub][_ord], "ip_int": _ints[_pub][_ord]})
print(f"{len(public)} unique public IPs  (of {len(_ips)} distinct IPv4 across all flows)")
print(public.isna().sum())
public

In [ ]:
# --- cell 10 ---
# Traffic type per session in vflowsAgg. Reuses `public` from the cell above, so
# public/private has one definition. Either end public -> north-south; both ends
# valid IPv4 and non-public -> east-west. Address-less / IPv6 rows stay <NA>.

_pub_ips = set(public["ip"])
_src_pub = vflowsAgg["srcaddr"].isin(_pub_ips)
_dst_pub = vflowsAgg["dstaddr"].isin(_pub_ips)
_src_v4  = vflowsAgg["srcaddr"].astype("string").str.count(r"\.").eq(3).fillna(False)
_dst_v4  = vflowsAgg["dstaddr"].astype("string").str.count(r"\.").eq(3).fillna(False)

vflowsAgg["type"] = pd.Series(pd.NA, index=vflowsAgg.index, dtype="string")
vflowsAgg.loc[_src_v4 & _dst_v4 & ~_src_pub & ~_dst_pub, "type"] = "east-west"
vflowsAgg.loc[_src_pub | _dst_pub, "type"] = "north-south"

print(vflowsAgg["type"].value_counts(dropna=False).to_string())

In [ ]:
# --- cell 11 ---
# Flow direction per session in vflowsAgg. v5 logs carry `flow-direction` (ENI-relative); v2/v3 don't, so
# derive it from which end is public: public dst = egress, public src = ingress.
# Reuses `public` from the cell above, so public/private has one definition here.
# DELETE THIS CELL once the source logs carry flow-direction -- everything
# downstream reads `direction` and nothing else derives it.

if "flow-direction" in vflowsAgg.columns:
    vflowsAgg["direction"] = vflowsAgg["flow-direction"]
    print("direction: taken from the log's flow-direction field")
else:
    _pub_ips = set(public["ip"])
    _src_pub = vflowsAgg["srcaddr"].isin(_pub_ips)
    _dst_pub = vflowsAgg["dstaddr"].isin(_pub_ips)

    vflowsAgg["direction"] = pd.Series(pd.NA, index=vflowsAgg.index, dtype="string")
    vflowsAgg.loc[_dst_pub & ~_src_pub, "direction"] = "egress"    # public destination
    vflowsAgg.loc[_src_pub & ~_dst_pub, "direction"] = "ingress"   # public source
    _v = pd.to_numeric(vflowsAgg["version"], errors="coerce").max()
    print(f"direction: DERIVED -- v{int(_v)} logs have no flow-direction field. "
          "internal<->internal, public<->public and address-less rows stay <NA>.")

print(vflowsAgg["direction"].value_counts(dropna=False).to_string())

In [ ]:
# --- cell 12 ---
# One interval index over AWS + Zscaler + Tor + Azure + Google (schema-identical frames).
# Vectorised CIDR -> [start, end]; one binary search per IP; most-specific prefix
# wins, so a Zscaler/Azure /24 beats a broad AWS /12. ~0.3 s for 53k prefixes.

if not (aws["ip_prefix"] == "72.21.192.0/19").any():
    aws = pd.concat([aws, pd.DataFrame([{
        "ip_prefix": "72.21.192.0/19", "region": "-",
        "service": "AMAZON", "network_border_group": "-"}])], ignore_index=True)

refs = pd.concat(
    [aws.assign(ref="aws"), zscaler.assign(ref="zscaler"), tor.assign(ref="tor"),
     azure.assign(ref="azure"), google.assign(ref="google")],
    ignore_index=True,
)[["ip_prefix", "service", "region", "ref"]]

def cidr_bounds(prefixes):
    """Vectorised 'a.b.c.d/n' -> (start_int, end_int) arrays, uint32 range in uint64."""
    p    = pd.Series(prefixes, dtype="string").str.split("/", n=1, expand=True)
    o    = p[0].str.split(".", expand=True).astype("uint64")
    base = (o[0] << np.uint64(24) | o[1] << np.uint64(16)
            | o[2] << np.uint64(8) | o[3]).to_numpy("uint64")
    plen = p[1].fillna("32").astype("uint64").to_numpy("uint64")
    mask = np.where(
        plen == 0, np.uint64(0),
        (np.uint64(0xFFFFFFFF) << (np.uint64(32) - plen)) & np.uint64(0xFFFFFFFF),
    ).astype("uint64")
    start = base & mask
    return start, start | (~mask & np.uint64(0xFFFFFFFF))

def match_cidr_index(idx, ip_ints):
    """Per IP: row index of the most specific containing prefix, or -1."""
    ip_ints = np.asarray(ip_ints, dtype="uint64")
    starts, ends, cummax = idx["starts"], idx["ends"], idx["cummax"]
    n    = len(starts)
    pos  = np.searchsorted(starts, ip_ints, side="right") - 1
    posc = np.clip(pos, 0, n - 1)
    out  = np.full(len(ip_ints), -1, dtype=np.int64)
    direct  = (pos >= 0) & (ends[posc]   >= ip_ints)
    covered = (pos >= 0) & (cummax[posc] >= ip_ints)
    out[direct] = pos[direct]
    for k in np.where(covered & ~direct)[0]:                        # rare: nested broad block
        j, ipv = int(pos[k]), ip_ints[k]
        while j >= 0 and ends[j] < ipv:
            j -= 1
        out[k] = j
    return out

_st, _en = cidr_bounds(refs["ip_prefix"])

# Sort by start asc, then end desc. For same-start overlaps, searchsorted(side="right")
# lands on the shortest containing prefix, i.e. the most-specific CIDR.

_o = np.lexsort((np.uint64(0xFFFFFFFF) - _en, _st))
ref_idx = {"starts": _st[_o], "ends": _en[_o],
           "cummax": np.maximum.accumulate(_en[_o]),
           "rows": refs.iloc[_o].reset_index(drop=True)}

public["ref_row"] = match_cidr_index(ref_idx, public["ip_int"].to_numpy("uint64"))
public["in_ref"]  = public["ref_row"] >= 0
_rr  = public["ref_row"].clip(lower=0).to_numpy()
_hit = public["in_ref"].to_numpy()
public["label"]        = np.where(_hit, ref_idx["rows"]["service"].to_numpy()[_rr],   np.nan)
public["label_source"] = np.where(_hit, ref_idx["rows"]["ref"].to_numpy()[_rr],       np.nan)
public["ref_prefix"]   = np.where(_hit, ref_idx["rows"]["ip_prefix"].to_numpy()[_rr], np.nan)
public["aws_region"]   = np.where(_hit & (public["label_source"].to_numpy() == "aws"),
                                  ref_idx["rows"]["region"].to_numpy()[_rr], np.nan)

print(public.loc[public["in_ref"], "label_source"].value_counts().to_string())
print(f"matched {int(public['in_ref'].sum())} / {len(public)} public IPs")
public.loc[public["in_ref"], ["ip", "ref_prefix", "label", "label_source"]].head()

In [ ]:
# --- cell 13 ---
# Known / trusted addresses from known_ips.txt (kept out of the notebook so it
# never gets committed). One IP or CIDR per line, optional label, # comments.
# Tags `public` with known / known_label, and writes known_label into `label`
# (label_source = "known") so it flows into vflowsAgg via the label cell. A known
# entry overrides a cloud label -- it's the more specific fact. Nothing is filtered.
# Missing / unreadable file -> warn and carry on with nothing known.

KNOWN_FILE = "known_ips.txt"

_rows = []
try:
    with open(KNOWN_FILE) as _f:
        for _ln in _f:
            _ln = _ln.split("#", 1)[0].split(None, 1)
            if _ln:
                _rows.append({"ip_prefix": _ln[0] if "/" in _ln[0] else _ln[0] + "/32",   # bare IP = /32
                              "label": _ln[1].strip() if len(_ln) > 1 else "known"})
except OSError as _e:
    print(f"!! {KNOWN_FILE} not loaded ({type(_e).__name__}: {_e}) -> nothing known")
known = pd.DataFrame(_rows, columns=["ip_prefix", "label"])

public["known"]       = False
public["known_label"] = pd.Series(pd.NA, index=public.index, dtype="string")
if len(known):
    _ks, _ke = cidr_bounds(known["ip_prefix"])
    _ip = public["ip_int"].to_numpy("uint64")
    for _s, _e, _l in zip(_ks, _ke, known["label"]):
        _hit = (_ip >= _s) & (_ip <= _e)
        public.loc[_hit, "known"]       = True
        public.loc[_hit, "known_label"] = _l

public.loc[public["known"], "label"]        = public.loc[public["known"], "known_label"]
public.loc[public["known"], "label_source"] = "known"

print(f"{len(known)} known entries -> {int(public['known'].sum())} public IPs tagged")

In [ ]:
# --- cell 14 ---
# Cloud-service label per session in vflowsAgg. Each public IP is looked up in both
# srcaddr and dstaddr; whichever end matches gives `label` from `public` (cell 09).
# Only one end should be public -- sessions where both are get counted below.

_lbl   = public.set_index("ip")["label"]
_src_m = vflowsAgg["srcaddr"].isin(_lbl.index)
_dst_m = vflowsAgg["dstaddr"].isin(_lbl.index)

vflowsAgg["label"] = (vflowsAgg["srcaddr"].where(_src_m).map(_lbl)
                      .combine_first(vflowsAgg["dstaddr"].where(_dst_m).map(_lbl))
                      .astype("string"))

_both = int((_src_m & _dst_m).sum())
if _both:
    print(f"!! {_both:,} sessions have a public IP at BOTH ends -- label taken from srcaddr")
print(f"{int((_src_m | _dst_m).sum()):,} of {len(vflowsAgg):,} sessions matched a public IP "
      f"(src {int(_src_m.sum()):,}, dst {int(_dst_m.sum()):,}); "
      f"{int(vflowsAgg['label'].notna().sum()):,} of those have a label")
print(vflowsAgg["label"].value_counts(dropna=False).head(15).to_string())

In [ ]:
# --- cell 15 ---
# Distinct reference prefixes that matched at least one public IP

_hit_rows   = np.unique(public.loc[public["in_ref"], "ref_row"].to_numpy())
refMatches = ref_idx["rows"].iloc[_hit_rows].reset_index(drop=True)
print(refMatches["ref"].value_counts().to_string())
refMatches

In [ ]:
# --- cell 16 ---
# label / label_source / aws_region are set by the combined-match cell above.
# (Precedence is most-specific-prefix-wins; AWS/Azure/Zscaler address space is
#  effectively disjoint, so label_source is unambiguous in practice.)

print(public["label_source"].value_counts(dropna=False).to_string())
print()
print(public["label"].value_counts(dropna=False).head(20).to_string())
public[["ip", "label", "label_source", "aws_region"]].dropna(subset=["label"]).head()

In [ ]:
# --- cell 17 ---
# Top sessions in vflowsAgg by bytes or by duration. Filter by type, direction, action, and whether
# the session has a cloud label. The table is an HTML widget whose .value is reassigned
# on every change (no Output widget, so nothing accumulates).

_CSS  = "font:13px system-ui,sans-serif;margin:6px 0"
# explicit cell padding: JupyterLab only pads its own DataFrame output, not HTML in widgets
_TBL_CSS = ("<style>.flowTbl{font:12px ui-monospace,monospace;border-collapse:collapse}"
            ".flowTbl th,.flowTbl td{padding:3px 12px;text-align:right;white-space:nowrap}"
            ".flowTbl thead th{border-bottom:1px solid #888}"
            ".flowTbl tbody tr:nth-child(even){background:rgba(128,128,128,.08)}</style>")
_wsty = {"description_width": "70px"}
_typW = widgets.ToggleButtons(options=["all", "north-south", "east-west"], description="type", style=_wsty)
_dirW = widgets.ToggleButtons(options=["all", "egress", "ingress"], description="direction", style=_wsty)
_actW = widgets.ToggleButtons(options=["all", "ACCEPT", "REJECT"], description="action", style=_wsty)
_lblW = widgets.ToggleButtons(options=["all", "labeled", "NA"], description="label", style=_wsty)
_srtW = widgets.ToggleButtons(options=["bytes", "duration"], description="sort by", style=_wsty)
_nW   = widgets.Dropdown(options=[25, 50, 100, 250], value=50, description="rows", style=_wsty,
                         layout=widgets.Layout(width="160px"))
_topTbl = widgets.HTML()

_topCols = ["start_time", "end_time", "hours", "srcaddr", "srcport", "dstaddr", "dstport",
            "protocol", "type", "action", "direction", "label", "mb", "packets", "records"]


def _topUpdate(_=None):
    m = pd.Series(True, index=vflowsAgg.index)
    if _typW.value != "all":
        m &= vflowsAgg["type"].eq(_typW.value).fillna(False)
    if _dirW.value != "all":
        m &= vflowsAgg["direction"].eq(_dirW.value).fillna(False)
    if _actW.value != "all":
        m &= vflowsAgg["action"].eq(_actW.value)
    if _lblW.value == "labeled":
        m &= vflowsAgg["label"].notna()
    elif _lblW.value == "NA":
        m &= vflowsAgg["label"].isna()

    sel = vflowsAgg[m]
    key = "bytes" if _srtW.value == "bytes" else "_dur"
    top = (sel.assign(_dur=sel["end"] - sel["start"]).dropna(subset=[key])
           .nlargest(_nW.value, key)
           .assign(mb=lambda d: (d["bytes"] / 1e6).round(2),
                   hours=lambda d: (d["_dur"] / 3600).round(2))
           [_topCols])
    _topTbl.value = (
        _TBL_CSS +
        f"<div style='{_CSS}'><b>{len(sel):,}</b> sessions, "
        f"<b>{sel['bytes'].sum() / 1e6:,.1f} MB</b> match &mdash; top {len(top)} by {_srtW.value}</div>"
        + top.to_html(index=False, border=0, escape=True, na_rep="-").replace(
            "<table", "<table class='flowTbl'"))


for _w in (_typW, _dirW, _actW, _lblW, _srtW, _nW):
    _w.observe(_topUpdate, names="value")

_topUpdate()
display(widgets.VBox([_typW, _dirW, _actW, _lblW, widgets.HBox([_srtW, _nW]), _topTbl]))

In [ ]:
# --- cell 18 ---
# Longest-running sessions in vflowsAgg (same filters as the top-bytes cell above,
# ranked by duration = end - start, and the summary sums hours instead of MB).
# Own widget names, so it doesn't share state with the cell above.

_CSS  = "font:13px system-ui,sans-serif;margin:6px 0"
# explicit cell padding: JupyterLab only pads its own DataFrame output, not HTML in widgets
_TBL_CSS = ("<style>.flowTbl{font:12px ui-monospace,monospace;border-collapse:collapse}"
            ".flowTbl th,.flowTbl td{padding:3px 12px;text-align:right;white-space:nowrap}"
            ".flowTbl thead th{border-bottom:1px solid #888}"
            ".flowTbl tbody tr:nth-child(even){background:rgba(128,128,128,.08)}</style>")
_wsty = {"description_width": "70px"}
_durTypW = widgets.ToggleButtons(options=["all", "north-south", "east-west"], description="type", style=_wsty)
_durDirW = widgets.ToggleButtons(options=["all", "egress", "ingress"], description="direction", style=_wsty)
_durActW = widgets.ToggleButtons(options=["all", "ACCEPT", "REJECT"], description="action", style=_wsty)
_durLblW = widgets.ToggleButtons(options=["all", "labeled", "NA"], description="label", style=_wsty)
_durNW   = widgets.Dropdown(options=[25, 50, 100, 250], value=50, description="rows", style=_wsty,
                            layout=widgets.Layout(width="160px"))
_durTbl  = widgets.HTML()

_durCols = ["start_time", "end_time", "hours", "srcaddr", "srcport", "dstaddr", "dstport",
            "protocol", "type", "action", "direction", "label", "mb", "packets", "records"]


def _durUpdate(_=None):
    m = pd.Series(True, index=vflowsAgg.index)
    if _durTypW.value != "all":
        m &= vflowsAgg["type"].eq(_durTypW.value).fillna(False)
    if _durDirW.value != "all":
        m &= vflowsAgg["direction"].eq(_durDirW.value).fillna(False)
    if _durActW.value != "all":
        m &= vflowsAgg["action"].eq(_durActW.value)
    if _durLblW.value == "labeled":
        m &= vflowsAgg["label"].notna()
    elif _durLblW.value == "NA":
        m &= vflowsAgg["label"].isna()

    sel = vflowsAgg[m].assign(hours=lambda d: (d["end"] - d["start"]) / 3600)
    top = (sel.dropna(subset=["hours"]).nlargest(_durNW.value, "hours")
           .assign(mb=lambda d: (d["bytes"] / 1e6).round(2),
                   hours=lambda d: d["hours"].round(2))
           [_durCols])
    _durTbl.value = (
        _TBL_CSS +
        f"<div style='{_CSS}'><b>{len(sel):,}</b> sessions, "
        f"<b>{sel['hours'].sum():,.1f} hours</b> match &mdash; top {len(top)} by duration</div>"
        + top.to_html(index=False, border=0, escape=True, na_rep="-").replace(
            "<table", "<table class='flowTbl'"))


for _w in (_durTypW, _durDirW, _durActW, _durLblW, _durNW):
    _w.observe(_durUpdate, names="value")

_durUpdate()
display(widgets.VBox([_durTypW, _durDirW, _durActW, widgets.HBox([_durLblW, _durNW]), _durTbl]))

In [ ]:
# --- cell 19 ---
# Beacon candidates: outbound, allowed traffic that checks in on a regular interval
# with small, steady payloads over a long period. Groups RAW records (vflows, not
# vflowsAgg -- merging loses the per-record timing) by local host -> remote addr,
# port, protocol, ignoring source port, so it catches both a held-open connection
# (one srcport) and one that reconnects every check-in (a new srcport each time).
# Caveat: flow logs bucket traffic into ~1-minute capture windows, so any steady
# connection produces ~60 s records -- the small-payload limits are what separate
# a beacon from a busy stream. Intervals much under a minute can't be resolved.

BEACON_INTERVAL_S = (25, 75)   # median check-in interval must fall in this range ("30-60 s, give or take")
BEACON_JITTER     = 0.25       # an interval counts as on-beat if within +/-25% of the median
BEACON_REGULAR    = 0.60       # ...and at least this share of intervals must be on-beat
BEACON_MAX_BYTES  = 2000       # median bytes per record (outbound) at most
BEACON_MAX_PKTS   = 20         # median packets per record (outbound) at most
BEACON_MIN_HOURS  = 6          # first-to-last span at least
BEACON_MIN_RECS   = 60         # outbound records at least

_pubIps = set(public["ip"])
_bk     = ["srcaddr", "dstaddr", "dstport", "protocol"]
_out    = vflows[vflows["action"].eq("ACCEPT") & vflows["dstaddr"].isin(_pubIps) & ~vflows["srcaddr"].isin(_pubIps)]
_back   = vflows[vflows["action"].eq("ACCEPT") & vflows["srcaddr"].isin(_pubIps) & ~vflows["dstaddr"].isin(_pubIps)]

# check-in intervals: gaps between distinct record start times per group
_t = _out[_bk + ["start"]].drop_duplicates().sort_values(_bk + ["start"])
_t["gap"] = _t.groupby(_bk)["start"].diff()
_t = _t.dropna(subset=["gap"])
_t["onBeat"] = ((_t["gap"] - _t.groupby(_bk)["gap"].transform("median")).abs()
                <= BEACON_JITTER * _t.groupby(_bk)["gap"].transform("median"))
_iv = _t.groupby(_bk).agg(interval_s=("gap", "median"), regular=("onBeat", "mean"))

_st = _out.groupby(_bk).agg(records=("bytes", "size"), srcports=("srcport", "nunique"),
                            first=("start_time", "min"), last=("end_time", "max"),
                            _s=("start", "min"), _e=("end", "max"),
                            bytesRec=("bytes", "median"), pktsRec=("packets", "median"),
                            bytesOut=("bytes", "sum"))
_in = (_back[["dstaddr", "srcaddr", "srcport", "protocol", "bytes"]]           # replies, flipped to the outbound key
       .set_axis(_bk + ["bytes"], axis=1)
       .groupby(_bk)["bytes"].sum().rename("bytesIn"))

beaconCandidates = (_st.join(_iv).join(_in)
    .assign(spanHours=lambda d: ((d["_e"] - d["_s"]) / 3600).round(1),
            kbOut=lambda d: (d["bytesOut"] / 1e3).round(1),
            kbIn=lambda d: (d["bytesIn"] / 1e3).round(1),
            regular=lambda d: d["regular"].astype(float).round(2))
    .drop(columns=["_s", "_e"])
    .reset_index())
beaconCandidates["label"] = beaconCandidates["dstaddr"].map(public.set_index("ip")["label"]).astype("string")

_lo, _hi = BEACON_INTERVAL_S
_hit = (beaconCandidates["interval_s"].between(_lo, _hi)
        & beaconCandidates["regular"].ge(BEACON_REGULAR)
        & beaconCandidates["bytesRec"].le(BEACON_MAX_BYTES)
        & beaconCandidates["pktsRec"].le(BEACON_MAX_PKTS)
        & beaconCandidates["spanHours"].ge(BEACON_MIN_HOURS)
        & beaconCandidates["records"].ge(BEACON_MIN_RECS))
beaconCandidates = (beaconCandidates[_hit]
    [["srcaddr", "dstaddr", "dstport", "protocol", "label", "interval_s", "regular",
      "bytesRec", "pktsRec", "records", "srcports", "spanHours", "first", "last", "kbOut", "kbIn"]]
    .sort_values("spanHours", ascending=False)
    .reset_index(drop=True))

print(f"interval {_lo}-{_hi} s, >= {BEACON_REGULAR:.0%} on-beat (+/-{BEACON_JITTER:.0%}), "
      f"<= {BEACON_MAX_BYTES} B and <= {BEACON_MAX_PKTS} pkts per record, "
      f">= {BEACON_MIN_HOURS} h, >= {BEACON_MIN_RECS} records")
print(f"{len(beaconCandidates)} beacon candidates of {len(_st):,} outbound host/peer/port groups")
beaconCandidates